# Continuity in street networks

Momepy allows for the deduction of natural continuity of street networks using the COINS algorithm. The street network is split into individual segments and deflection angles between adjacent segments are computed. Segments are then joined to continuous strokes. Segments will only be considered a part of the same stroke if the deflection angle is above the threshold (defaults to zero).

## Using a small network

In [ ]:
import geopandas as gpd
import momepy

In [ ]:
streets = gpd.read_file(momepy.datasets.get_path("bubenec"), layer="streets")

In [ ]:
streets.plot(figsize=(10, 10)).set_axis_off()

`momepy.COINS` allows you to create a `GeoDataFrame` of the final merged strokes or return a `Series` with a stroke attribute which can be attached to the original geometry.

You first need to compute continuity using the `momepy.COINS` class.

In [ ]:
continuity = momepy.COINS(streets)

Now you can generate required outputs.

In [ ]:
stroke_gdf = continuity.stroke_gdf()
stroke_gdf

We can plot the data using an unique color per stroke geometry.

In [ ]:
stroke_gdf.plot(cmap="tab10", figsize=(10, 10)).set_axis_off()

In [ ]:
streets["continuity_stroke"] = continuity.stroke_attribute()

In [ ]:
streets.head()

In [ ]:
streets.plot(
    "continuity_stroke", categorical=True, figsize=(10, 10)
).set_axis_off()

## Using OpenStreetMap data

In [ ]:
import osmnx as ox

In [ ]:
streets_graph = ox.graph_from_place(
    "Vicenza, Vicenza, Italy", network_type="drive"
)
streets_graph = ox.projection.project_graph(streets_graph)

streets = ox.graph_to_gdfs(
    ox.convert.to_undirected(streets_graph),
    nodes=False,
    edges=True,
    node_geometry=False,
    fill_edge_geometry=True,
)

In [ ]:
streets.plot(figsize=(10, 10), linewidth=0.2).set_axis_off()

In [ ]:
continuity = momepy.COINS(streets)

In [ ]:
stroke_gdf = continuity.stroke_gdf()

We can look into the continuity-based hierarchy of streets based on their total length.

In [ ]:
stroke_gdf.plot(
    stroke_gdf.length,
    figsize=(15, 15),
    cmap="viridis_r",
    linewidth=0.5,
    scheme="headtailbreaks",
).set_axis_off()

For details, see the original paper:

Tripathy, P., Rao, P., Balakrishnan, K., & Malladi, T. (2020). An open-source tool to extract natural continuity and hierarchy of urban street networks. _Environment and Planning B: Urban Analytics and City Science_. http://dx.doi.org/10.1177/2399808320967680